# Class 17: Boundary Value Problems 

Ch. 8, section 6: Boundary Value and Eigenvalue Problems 

- Complete the activities as instructed by the professor

#### Import any packages we need below (update as we go):

In [1]:
import numpy as np
import matplotlib.pyplot as plt

### Shooting Method

#### Book example 8.8 - Vertical position of a thrown ball

Let's solve the problem above with the thrown ball for the case where the ball lands back at $x=0$ after $t = 10$ seconds. 
$$\frac{d^2x}{dt^2} = -g $$

**First step**: break down second-order differential equation into two first-orders:


$$ \frac{dx}{dt} = y \qquad \frac{dy}{dt} = -g $$

**Solve** For the initial velocity using Runge-Kutta and Binary Search 

Binary Search Outline: 
1) Given an initial pair of points $x_1, x_2$ check that $f(x_1)$ and $f(x_2)$ have opposite signs. Also choose a target accuracy for the answer you want. 

2) Calculate the midpoint $x' = \frac{1}{2}(x_1+x_2)$ and evaluate $f(x')$

3) If $f(x')$ has the same sign as $f(x_1)$ then set $x_1 = x'$. Otherwise set $x_2 = x'$. 

4) If $|{x_1-x_2}|$ is greater than the target accuracy, repeat from step 2. Otherwise, calculate $\frac{1}{2}(x_1+x_2)$ once more and this is the final estimate of the position of the root. 


In [ ]:
# define functions for the differential equations
def f(r,t):
    g = 9.81 # m/s^2
    x = r[0]
    y = r[1]
    fx = y
    fy = -g
    return np.array([fx,fy], float)

# apply 4th order runge-kutta
def height(v,tpts,h):
    # tpts = array of time values spaced by h
    # v = initial velocity
    r = np.array([0.0, v], float)
    for t in tpts:
        k1 = h*f(r,t)
        k2 = h*f(r+0.5*k1, t+0.5*h)
        k3 = h*f(r+0.5*k2, t+0.5*h)
        k4 = h*f(r+k3, t+h)
        r += (k1 + 2*k2 + 2*k3 + k4) / 6
    return r[0]

# define range of values
t1 = 0.0
t2 = 10.0
N = 1000
h = (t2-t1)/N
tpts = np.arange(t1,t2+h,h)


# initial of value

# apply binary search to find the root
# choose target accuracy
target = 1e-10
v1 = 0.01
v2 = 1000.0
h1 = height(v1,tpts,h)
h2 = height(v2,tpts,h)
print(h1,h2)

# binary search method
while abs(h2-h1) > target:
    vp = (v1+v2)/2
    hp = height(vp,tpts, h)
    if h1*hp > 0:
        v1 = vp
        h1 = hp
    else:
        v2 = vp
        h2 = hp

v = (v1+v2)/2
print("the required initial velocity is", round(v,2),"m/s")

-491.3813905000039 9518.518509499776
the required initial velocity is 49.1 m/s


**Check** Plug back in as initial condition using Runge-Kutta to double-check final position 

In [23]:

# define range of values
t1 = 0.0
t2 = 10.0
N = 1000
h = (t2-t1)/N
tpts = np.arange(t1,t2+h,h)
xpts = np.zeros(len(tpts)) # array of heights
ypts = np.zeros(len(tpts)) # array of velocities

# initial of value

r = np.array([0.0, v], float)
    
for i in range(len(tpts)):
    
    k1 = h*f(r, tpts[i])
    k2 = h*f(r+0.5*k1, tpts[i]+0.5*h)
    k3 = h*f(r+0.5*k2, tpts[i]+0.5*h)
    k4 = h*f(r+k3, tpts[i]+h)
    r += (k1 + 2*k2 + 2*k3 + k4) / 6
    xpts[i] = r[0]
    ypts[i] = r[1]


print(xpts[-1])

3.223127320595154e-11


#### What if...

What if the final condition on position had been a value rather than 0? 

**answer**: Since searching for when the `h1` and `h2` values straddle a root, just subtract the final height from each of them to make it a zero

### Eigenvalue Problems

Calculate the ground state energy of an electron in a square potential well with infinitely high walls separated by a distance $L$ equal to the Bohr radius $a_0 = 5.292 \times 10^{-11}$ m.

$$ \frac{d\psi}{dx} = \phi \qquad \frac{d\phi}{dx} = \frac{2m}{\hbar^2}\left[ V(x) -E \right] \psi $$

**Secant method - solution to noninear equations**

new, hopefully better estimate of root is given by: 
$$x_3 = x_2 - f(x_2) \frac{x_1-x_2}{f(x_1)-f(x_2)}$$
where the new estimate is based on the previous two values in the series 

In [ ]:
# potential energy
def V(x):
    return 0.0

# differential equation functions
def f(r,x,E):
    psi = r[0]
    phi = r[1]
    fpsi = phi
    fphi = (2*m/hbar**2)*(V(x) - E)*psi
    return np.array([fpsi,fphi], float)

# constants
m = 9.1094e-31 # mass of an electron
hbar = 1.0546e-34 # planck's constant over 2pi
e = 1.6022e-19 # electron charge
L = 5.2918e-11 # Bohr radius 
N = 1000
h = L/N

# apply RK4 to calculate wavefunction for particular energy
def wavefn_RK4(E):
    r = np.array([0.0,1.0], float) # initial guesses for psi, phi

    # dependant of x, not t
    xpts = np.arange(0, L+h, h)

    for x in xpts:
        k1 = h*f(r,x,E)
        k2 = h*f(r+0.5*k1, x+0.5*h,E)
        k3 = h*f(r+0.5*k2, x+0.5*h,E)
        k4 = h*f(r+k3, x+h,E)
        r += (k1 + 2*k2 + 2*k3 + k4) / 6
    return r[0]

# determine energy using secant method
E1 = 0.0
E2 = e
psi2 = wavefn_RK4(E1)

target = e/1000
while abs(E1-E2)>target:
    psi1,psi2 = psi2,wavefn_RK4(E2)
    E1,E2 = E2,E2 - psi2*(E1-E2)/(psi1-psi2) # not many steps!

print("E =", E2/e, "eV")




E = 134.0182012729446 eV


### Exercise 8.14(a) - if time

Consider the one-dimensional, time-independent Schroedinger equation in a harmonic (i.e., quadratic) potential $V(x) = V_0x^2/a^2$, where $V_0$ and $a$ are constants.

**a)** Write down the Schroedinger equation for this problem and convert it from a second- order equation to two first-order ones, as in Example 8.9. Write a program, or modify the one from Example 8.9, to find the energies of the ground state and the first two excited states for these equations when $m$ is the electron mass, $V_0 = 50$ eV, and $a = 10^{−11}$ m. Note that in theory the wavefunction goes all the way out to $x = \pm \infty$, but you can get good answers by using a large but finite interval. Try using $x = −10a$ to $+10a$, with the wavefunction $\psi = 0$ at both boundaries. (In effect, you are putting the harmonic oscillator in a box with impenetrable walls.) The wavefunction is real everywhere, so you don’t need to use complex variables, and you can use evenly spaced points for the solution—there is no need to use an adaptive method for this problem.

The quantum harmonic oscillator is known to have energy states that are equally spaced. Check that this is true, to the precision of your calculation, for your answers. (Hint: The ground state has energy in the range 100 to 200 eV.)

In [ ]:



# constants
m = 9.1094e-31 # mass of an electron
hbar = 1.0546e-34 # planck's constnat over 2pi
e = 1.6022e-19 # electron charge
V0 = 50*e